# ====================== Quraşdırma ======================


In [ ]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor, Seq2SeqTrainingArguments, Seq2SeqTrainer
from peft import LoraConfig, get_peft_model
from datasets import Dataset
import pandas as pd, os, librosa, torch, re, json
from evaluate import load
import matplotlib.pyplot as plt
from tqdm import tqdm
from dataclasses import dataclass
from typing import Any, Dict, List
import glob


# =================== Konfiqurasiya ====================


In [ ]:
MODEL_NAME = "LocalDoc/azerbaijani-whisper-small"
DATA_PATH = "../az/"
CLIPS_DIR = os.path.join(DATA_PATH, "clips")
OUTPUT_DIR = "./whisper-small-az-lora"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# =================== Model ====================


In [ ]:
processor = WhisperProcessor.from_pretrained(MODEL_NAME)
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME).to(DEVICE)

model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"], bias="none"
))

# =================== Data hazırlığı ====================


In [ ]:
def prepare_dataset(split):
    df = pd.read_csv(os.path.join(DATA_PATH, f"{split}.tsv"), sep='\t')
    ds = Dataset.from_pandas(df[['path', 'sentence']])
    
    def preprocess(batch):
        audio_path = os.path.join(CLIPS_DIR, batch["path"])
        speech, _ = librosa.load(audio_path, sr=16000)
        return {
            "input_features": processor(speech, sampling_rate=16000).input_features[0],
            "labels": processor(text=batch["sentence"]).input_ids
        }
    
    return ds.map(preprocess, remove_columns=ds.column_names)

train_dataset = prepare_dataset("train")
eval_dataset = prepare_dataset("dev")

print(f"Train: {len(train_dataset)} | Eval: {len(eval_dataset)}")

# ==================== Data Collator ====================


In [ ]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        batch = self.processor.feature_extractor.pad(
            [{"input_features": f["input_features"]} for f in features], return_tensors="pt"
        )
        labels_batch = self.processor.tokenizer.pad(
            [{"input_ids": f["labels"]} for f in features], return_tensors="pt"
        )
        batch["labels"] = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        return batch

# ================== Training Arguments ==================


In [ ]:
wer_metric = load("wer")
cer_metric = load("cer")

def compute_metrics(pred):
    pred_str = processor.batch_decode(pred.predictions, skip_special_tokens=True)
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)
    
    return {
        "wer": wer_metric.compute(predictions=pred_str, references=label_str),
        "cer": cer_metric.compute(predictions=pred_str, references=label_str)
    }

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=4,
    learning_rate=1e-4,
    max_steps=150,
    eval_steps=20,
    save_steps=20,
    logging_steps=10,
    
    save_total_limit=1,             
    load_best_model_at_end=True,     
    metric_for_best_model="wer",     
    greater_is_better=False,         
    
    eval_strategy="steps",
    save_strategy="steps",
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model, args=training_args,
    train_dataset=train_dataset, eval_dataset=eval_dataset,
    data_collator=DataCollatorSpeechSeq2SeqWithPadding(processor=processor),
    processing_class=processor.feature_extractor,
    compute_metrics=compute_metrics,
)

# ================== Fine-tuning ==================


In [ ]:
print("\nFine-tuning başlayır...")
trainer.train()

# ================== B3: MÜQAYİSƏ ==================


In [ ]:
print("\n Müqayisə başlayır...")

def transcribe(model, audio_path):
    speech, _ = librosa.load(audio_path, sr=16000)
    input_features = processor(speech, sampling_rate=16000, return_tensors="pt").input_features.to(model.device)
    with torch.no_grad():
        predicted_ids = model.generate(input_features, max_new_tokens=256, language="az", task="transcribe")
    return processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

def normalize_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zəöüğıçş\s]', '', text)
    return re.sub(r'\s+', ' ', text).strip()

test_df = pd.read_csv(os.path.join(DATA_PATH, "test.tsv"), sep='\t')
test_df['audio_path'] = test_df['path'].apply(lambda x: os.path.join(CLIPS_DIR, x))

base_model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME).to(DEVICE)

base_preds, ft_preds, refs = [], [], []
for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    refs.append(row['sentence'])
    base_preds.append(transcribe(base_model, row['audio_path']))
    ft_preds.append(transcribe(model, row['audio_path']))


clean_base = [normalize_text(p) for p in base_preds]
clean_ft = [normalize_text(p) for p in ft_preds]
clean_ref = [normalize_text(r) for r in refs]

wer_base = wer_metric.compute(predictions=clean_base, references=clean_ref)
cer_base = cer_metric.compute(predictions=clean_base, references=clean_ref)
wer_ft = wer_metric.compute(predictions=clean_ft, references=clean_ref)
cer_ft = cer_metric.compute(predictions=clean_ft, references=clean_ref)

results = pd.DataFrame({
    "Model": ["Baza Model", "Fine-tuned Model"],
    "WER (%)": [round(wer_base*100, 2), round(wer_ft*100, 2)],
    "CER (%)": [round(cer_base*100, 2), round(cer_ft*100, 2)]
})

print("\n" + "="*60)
print("NƏTİCƏLƏR")
print("="*60)
print(f"Baza Model      | WER: {wer_base*100:.2f}% | CER: {cer_base*100:.2f}%")
print(f"Fine-tuned Model| WER: {wer_ft*100:.2f}%  | CER: {cer_ft*100:.2f}%")
print("="*60)

os.makedirs("../results", exist_ok=True)
results.to_csv("../results/b3_model_comparison.csv", index=False)

# ================== B3: QRAFİKLƏR ==================


In [ ]:
checkpoint_files = sorted(glob.glob(f"{OUTPUT_DIR}/checkpoint-*/trainer_state.json"))
history = []
for f in reversed(checkpoint_files):
    try:
        with open(f, "r") as fp:
            history = json.load(fp).get("log_history", [])
        if history:
            print(f"İstifadə olunan checkpoint: {os.path.basename(os.path.dirname(f))}")
            break
    except:
        continue

if history:
    train_steps, train_losses = [], []
    eval_steps, eval_losses, eval_wers, eval_cers = [], [], [], []
    
    for entry in history:
        step = entry.get('step', 0)
        if 'loss' in entry and 'eval_loss' not in entry:
            train_steps.append(step)
            train_losses.append(entry['loss'])
        if 'eval_loss' in entry:
            eval_steps.append(step)
            eval_losses.append(entry['eval_loss'])
        if 'eval_wer' in entry:
            eval_wers.append(entry['eval_wer'])
        if 'eval_cer' in entry:
            eval_cers.append(entry['eval_cer'])
    
    if train_losses and eval_losses:
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))
        
        ax1.plot(train_steps, train_losses, 'b-o', markersize=5, label='Training Loss')
        ax1.plot(eval_steps, eval_losses, 'r-s', markersize=7, label='Validation Loss')
        ax1.set_title('Loss (Overfitting Monitorinqi)', fontweight='bold')
        ax1.set_xlabel('Addım'); ax1.set_ylabel('Loss')
        ax1.legend(); ax1.grid(True, alpha=0.3)
        
        if eval_wers:
            ax2.plot(eval_steps, eval_wers, 'orange', marker='D', markersize=8, linewidth=2)
            ax2.set_ylabel('WER', color='orange')
            ax2.tick_params(axis='y', labelcolor='orange')
            
            min_wer = min(eval_wers)
            best_step = eval_steps[eval_wers.index(min_wer)]
            ax2.scatter(best_step, min_wer, s=150, c='red', zorder=5)
            ax2.annotate(f'Best: {min_wer:.3f}', (best_step, min_wer), xytext=(10, -15), textcoords="offset points", color='red', fontweight='bold')
        
        # CER qrafiki
        if eval_cers:
            ax2b = ax2.twinx()
            ax2b.plot(eval_steps, eval_cers, 'purple', marker='s', markersize=8, linewidth=2)
            ax2b.set_ylabel('CER', color='purple')
            ax2b.tick_params(axis='y', labelcolor='purple')
            
            min_cer = min(eval_cers)
            best_cer_step = eval_steps[eval_cers.index(min_cer)]
            ax2b.scatter(best_cer_step, min_cer, s=150, c='blue', zorder=5)
            ax2b.annotate(f'Best: {min_cer:.3f}', (best_cer_step, min_cer),xytext=(10, 15), textcoords="offset points", color='blue', fontweight='bold')
        
        ax2.set_title('Validation WER & CER', fontweight='bold')
        ax2.set_xlabel('Addım'); ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        os.makedirs("../results", exist_ok=True)
        plt.savefig("../results/training_progress.png", dpi=200, bbox_inches='tight')
        plt.show()
        

        print("TRAİNİNG ANALİZİ")

        print(f"Train Loss: {train_losses[0]:.3f} → {train_losses[-1]:.3f}")
        print(f"Val Loss:  {eval_losses[0]:.3f} → {eval_losses[-1]:.3f}")
        print(f" Best WER:  {min(eval_wers):.3f}" if eval_wers else " WER: Məlumat yoxdur")
        if eval_cers:
            print(f" Best CER:  {min(eval_cers):.3f}")
        
        diff = abs(train_losses[-1] - eval_losses[-1])
        status = " YOXDUR" if diff < 0.3 else " RİSK" if diff < 0.6 else " VAR"
        print(f" Overfitting: {status} (Δ={diff:.2f})")
    else:
        print(" Kifayət qədər məlumat yoxdur")
else:
    print("Heç bir trainer_state.json tapılmadı!")

print("\n Bütün proses tamamlandı!")
print(" Nəticələr: ../results/ qovluğunda")